# Skypt python do zaagregowania danych i ich oczyszczenia

Potrzebne do przeprowadzenia eksperymentu na narzędziu Open AI - chat gpt

#### 1. Importuje potrzebne biblioteki

In [1]:
import pandas as pd
import os

#### 2. Wczytuje wszystkie potrzebne pliki

In [2]:
folder_path = "..\dane\\tabela-dane"
csv_files = [files for files in os.listdir(folder_path) if files.endswith('.csv')]

#### 3. Tworzę listę DataFrame z wczytanych plików csv, którą następnie łączę w jeden DataFrame

In [3]:
df_list = []
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)

#### 4. Usuwam starą kolumnę "Id", która ma złe wartości

In [4]:
if "Id" in combined_df.columns:
    combined_df = combined_df.drop(columns=["Id"])

#### 5. Patrzę, jak wygląda struktura mojej tabeli

In [5]:
combined_df.head()

,Title,Senior,Company,Location,Salary,Currency,date
0,Senior Python Computer Vision Engineer,True,Unitem,"Wrocław, PL",15 000 – 20 000 PLN,PLN,01.01.2023
1,Senior/Lead Data Scientist,True,Addepto,Zdalnie + 5,18 480 – 35 280 PLN,PLN,01.01.2023
2,Mid/Senior Python AI Engineer,True,Vestigit,"Wrocław, PL",17 000 – 22 000 PLN,PLN,01.01.2023
3,AI Developer with ROS,False,Four Point,"Wrocław, PL",15 000 – 28 000 PLN,PLN,01.01.2023
4,NLP Developer,False,Talkie.ai,Zdalnie,12 000 – 20 000 PLN,PLN,01.01.2023


In [6]:
combined_df.shape

(528, 7)

Zatem wszystkie obserwacje zostały pomyślnie zaagregowane

#### 6. Dodaję nową kolumnę "Id"

In [7]:
combined_df["Id"] = range(1, len(combined_df) + 1)

In [8]:
combined_df.head()

,Title,Senior,Company,Location,Salary,Currency,date,Id
0,Senior Python Computer Vision Engineer,True,Unitem,"Wrocław, PL",15 000 – 20 000 PLN,PLN,01.01.2023,1
1,Senior/Lead Data Scientist,True,Addepto,Zdalnie + 5,18 480 – 35 280 PLN,PLN,01.01.2023,2
2,Mid/Senior Python AI Engineer,True,Vestigit,"Wrocław, PL",17 000 – 22 000 PLN,PLN,01.01.2023,3
3,AI Developer with ROS,False,Four Point,"Wrocław, PL",15 000 – 28 000 PLN,PLN,01.01.2023,4
4,NLP Developer,False,Talkie.ai,Zdalnie,12 000 – 20 000 PLN,PLN,01.01.2023,5


Zamieniam teraz kolejność kolumn w DataFrame

In [9]:
columns = ["Id"] + [column for column in combined_df.columns if column != "Id"]
combined_df = combined_df[columns]

In [10]:
combined_df.head()

,Id,Title,Senior,Company,Location,Salary,Currency,date
0,1,Senior Python Computer Vision Engineer,True,Unitem,"Wrocław, PL",15 000 – 20 000 PLN,PLN,01.01.2023
1,2,Senior/Lead Data Scientist,True,Addepto,Zdalnie + 5,18 480 – 35 280 PLN,PLN,01.01.2023
2,3,Mid/Senior Python AI Engineer,True,Vestigit,"Wrocław, PL",17 000 – 22 000 PLN,PLN,01.01.2023
3,4,AI Developer with ROS,False,Four Point,"Wrocław, PL",15 000 – 28 000 PLN,PLN,01.01.2023
4,5,NLP Developer,False,Talkie.ai,Zdalnie,12 000 – 20 000 PLN,PLN,01.01.2023


#### 7. Oczyśczmy kolumnę "Location" ze zbędnych znaków

Zacznę od usuwania kodów narodowych

In [11]:
combined_df['Location'] = combined_df['Location'].str.replace(',', '', regex=False)
combined_df['Location'] = combined_df['Location'].str.replace('PL', '', regex=False)
combined_df['Location'] = combined_df['Location'].str.replace('pl', '', regex=False)
combined_df['Location'] = combined_df['Location'].str.replace('Pl', '', regex=False)

In [12]:
combined_df['Location'] = combined_df['Location'].str.replace('NL', '', regex=False)
combined_df['Location'] = combined_df['Location'].str.replace('SA', '', regex=False)
combined_df['Location'] = combined_df['Location'].str.replace('HU', '', regex=False)
combined_df['Location'] = combined_df['Location'].str.replace('DK', '', regex=False)

Teraz usuwam liczby przy wartości "Zdalnie"

In [13]:
for numer in range(1, 9):
    combined_df['Location'] = combined_df['Location'].str.replace(f'\xa0+{numer}', '', regex=False)

Sprawdźmy, co jeszcze należy oczyścić

In [14]:
print(combined_df['Location'].unique())

['Wrocław ' 'Zdalnie  + 5' 'Zdalnie' 'Zdalnie  + 2' 'Zdalnie  + 1'
 'Kraków   + 2' 'Zdalnie  + 8' 'Kraków ' 'Taastrup ' 'Warszawa '
 'Kraków   + 1' 'Budapest ' 'Zdalnie   ' 'Warszawa' 'Wrocław' 'Warsaw'
 'Warszawa   ' 'Lublin   ' 'Kraków   ' 'Sopot   ' 'Budapest   '
 'Amsterdam   ' 'Ołtarzew   ' 'Łódź' 'Kraków' 'Olsztyn' 'Warsaw   '
 'Poznań' 'Szolnok   ' 'Gliwice' 'Otwock' 'Katowice' 'Riyadh     ']


Pozostaje usunąć puste znaki z wartości i zamiana "Warsaw" na "Warszawa"

In [15]:
combined_df['Location'] = combined_df['Location'].str.strip()

In [16]:
 combined_df['Location'] = combined_df['Location'].str.replace('Warsaw', 'Warszawa', regex=False)

In [17]:
print(combined_df['Location'].unique())

['Wrocław' 'Zdalnie  + 5' 'Zdalnie' 'Zdalnie  + 2' 'Zdalnie  + 1'
 'Kraków   + 2' 'Zdalnie  + 8' 'Kraków' 'Taastrup' 'Warszawa'
 'Kraków   + 1' 'Budapest' 'Lublin' 'Sopot' 'Amsterdam' 'Ołtarzew' 'Łódź'
 'Olsztyn' 'Poznań' 'Szolnok' 'Gliwice' 'Otwock' 'Katowice' 'Riyadh']


#### 7. Rozdzielenie kolumny "Salary" na dwie osobne: Salary_min i Salary_max

In [18]:
combined_df[['Salary_min', 'Salary_max']] = combined_df['Salary'].str.split(' – ', expand=True)

In [19]:
print(combined_df['Salary'])

0      15 000  – 20 000  PLN
1      18 480  – 35 280  PLN
2      17 000  – 22 000  PLN
3      15 000  – 28 000  PLN
4      12 000  – 20 000  PLN
               ...          
523    18 000  – 26 000  PLN
524    20 000  – 28 000  PLN
525     9 000  – 14 000  PLN
526    15 000  – 16 000  PLN
527      8 000  – 9 000  PLN
Name: Salary, Length: 528, dtype: object


In [20]:
print(combined_df['Salary_min'])

0      15 000 
1      18 480 
2      17 000 
3      15 000 
4      12 000 
        ...   
523    18 000 
524    20 000 
525     9 000 
526    15 000 
527     8 000 
Name: Salary_min, Length: 528, dtype: object


In [21]:
print(combined_df['Salary_max'])

0      20 000  PLN
1      35 280  PLN
2      22 000  PLN
3      28 000  PLN
4      20 000  PLN
          ...     
523    26 000  PLN
524    28 000  PLN
525    14 000  PLN
526    16 000  PLN
527     9 000  PLN
Name: Salary_max, Length: 528, dtype: object


Teraz należy usunąć kolumnę "Salary" oraz oczyścić otrzymane kolumny:

In [22]:
if "Salary" in combined_df.columns:
    combined_df = combined_df.drop(columns=["Salary"])

In [23]:
combined_df['Salary_min'] = combined_df['Salary_min'].str.replace('  PLN', '', regex=False)
combined_df['Salary_max'] = combined_df['Salary_max'].str.replace('  PLN', '', regex=False)

In [24]:
combined_df['Salary_min'] = combined_df['Salary_min'].str.strip()
combined_df['Salary_max'] = combined_df['Salary_max'].str.strip()

In [25]:
print(combined_df['Salary_max'].unique())

['20\xa0000' '35\xa0280' '22\xa0000' '28\xa0000' '19\xa0000' '27\xa0000'
 '26\xa0000' '30\xa0827' '38\xa0640' '35\xa0700' '47\xa0000' '450\xa0000'
 '28\xa0500' '30\xa0000' '35\xa0000' '18\xa0000' '22\xa0680' '23\xa0000'
 '51\xa0379' '15\xa0000' '39\xa0573' '8\xa0000' '25\xa0000' '9\xa0000'
 '12\xa0989' '29\xa0400' '13\xa0100' '24\xa0000' '33\xa0000' '35\xa0500'
 '70\xa0000' '21\xa0000' '60\xa0000' '74\xa0600' None '23\xa0520'
 '33\xa0600' '119\xa0000' '25\xa0200' '29\xa0604' '37\xa0099' '3\xa0675'
 '34\xa0000' '38\xa0000' '12\xa0000' '27\xa0905' '180\xa0811' '22\xa0429'
 '32\xa0000' '21\xa0840' '36\xa0325' '22\xa0118' '24\xa0576' '25\xa0132'
 '35\xa0394' '17\xa0000' '33\xa0509' '14\xa0000' '44\xa0790' '37\xa0000'
 '16\xa0000' '19\xa0430' '17\xa0001' '24\xa0288' '9\xa0715' '18\xa0216'
 '31\xa0574' '20\xa0160' '40\xa0869' '40\xa0000' '24\xa0521' '34\xa0534'
 '29\xa0417' '40\xa0810' '24\xa0486' '19\xa0900' '11\xa0760' '20\xa0498'
 '28\xa0470' '10\xa0000' '31\xa0500' '34\xa0867' '29\xa0000

In [26]:
combined_df['Salary_min'] = combined_df['Salary_min'].str.replace('\\xa0', '', regex=False)
combined_df['Salary_max'] = combined_df['Salary_max'].str.replace('\\xa0', '', regex=False)
    
combined_df['Salary_min'] = combined_df['Salary_min'].str.strip()
combined_df['Salary_max'] = combined_df['Salary_max'].str.strip()

In [27]:
print(combined_df['Salary_max'].unique())

['20\xa0000' '35\xa0280' '22\xa0000' '28\xa0000' '19\xa0000' '27\xa0000'
 '26\xa0000' '30\xa0827' '38\xa0640' '35\xa0700' '47\xa0000' '450\xa0000'
 '28\xa0500' '30\xa0000' '35\xa0000' '18\xa0000' '22\xa0680' '23\xa0000'
 '51\xa0379' '15\xa0000' '39\xa0573' '8\xa0000' '25\xa0000' '9\xa0000'
 '12\xa0989' '29\xa0400' '13\xa0100' '24\xa0000' '33\xa0000' '35\xa0500'
 '70\xa0000' '21\xa0000' '60\xa0000' '74\xa0600' None '23\xa0520'
 '33\xa0600' '119\xa0000' '25\xa0200' '29\xa0604' '37\xa0099' '3\xa0675'
 '34\xa0000' '38\xa0000' '12\xa0000' '27\xa0905' '180\xa0811' '22\xa0429'
 '32\xa0000' '21\xa0840' '36\xa0325' '22\xa0118' '24\xa0576' '25\xa0132'
 '35\xa0394' '17\xa0000' '33\xa0509' '14\xa0000' '44\xa0790' '37\xa0000'
 '16\xa0000' '19\xa0430' '17\xa0001' '24\xa0288' '9\xa0715' '18\xa0216'
 '31\xa0574' '20\xa0160' '40\xa0869' '40\xa0000' '24\xa0521' '34\xa0534'
 '29\xa0417' '40\xa0810' '24\xa0486' '19\xa0900' '11\xa0760' '20\xa0498'
 '28\xa0470' '10\xa0000' '31\xa0500' '34\xa0867' '29\xa0000

In [30]:
combined_df.to_csv("../eksperyment/chat-gpt/dane.csv", index=False)